# Entregable 3


## 1. Importación de librerías

In [1]:
!pip install textblob spicy
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
from textblob import TextBlob

%matplotlib inline


### 1.1. Instalación de TextBlob

In [2]:
!pip install textblob
from textblob import TextBlob

## 2. Carga de los datasets

Se cargan los dos archivos indicados en la consigna.  
En esta notebook se asume que los archivos están en el mismo directorio que la notebook o se ajusta la ruta en las variables `train_path` y `test_path`.

Columnas según la descripción de la cátedra:

0 - `sentiment`: polaridad del tweet (0 = negativo, 2 = neutral, 4 = positivo)  
1 - `id`: id del tweet  
2 - `date`: fecha del tweet  
3 - `query`: término de búsqueda (o `NO_QUERY`)  
4 - `user`: nombre de usuario  
5 - `text`: texto del tweet  


### Carga corregida de datasets (big train / small test)

In [3]:
cols = ["sentiment", "id", "date", "query", "user", "text"]
train_path = "training.1600000.processed.noemoticon.csv"
ext_path   = "testdata.manual.2009.06.14.csv"

big_df = pd.read_csv(train_path, encoding="latin-1", names=cols)
ext_df = pd.read_csv(ext_path,   encoding="latin-1", names=cols)

big_df['sentiment'] = big_df['sentiment'].astype(int)
ext_df['sentiment']  = ext_df['sentiment'].astype(int)

print("big_df:", big_df.shape, "ext_df:", ext_df.shape)
big_df.head()


big_df: (1600000, 6) ext_df: (498, 6)


,sentiment,id,date,query,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


### 2.1. Distribucion inicial de clases en el dataset grande (train/test interno)


In [4]:
big_df['sentiment'].value_counts()


sentiment
0    800000
4    800000
Name: count, dtype: int64

### 2.2. Distribucion de clases en el dataset de validacion externa (dataset chico)


In [5]:
ext_df['sentiment'].value_counts()


sentiment
4    182
0    177
2    139
Name: count, dtype: int64

## 3. Preprocesamiento de texto (inglés)

Los tweets están en inglés, por lo que el preprocesamiento se adapta a este idioma.

**Decisiones de limpieza:**

- Convertir todo a minúsculas para evitar duplicar palabras por diferencias de mayúsculas/minúsculas.  
- Eliminar menciones (`@usuario`) y URLs, ya que no aportan información directa de sentimiento.  
- Conservar solo palabras en inglés y **contracciones** (por ejemplo, `can't`, `don't`, `i'm`).  
  Esto evita errores como convertir `can't` en `can t`, que generaría tokens artificiales (`can`, `t`).  
- No se eliminan stopwords explícitamente; se confía en que TF-IDF reduzca su peso.


In [6]:
def limpiar_tweet(texto):
    texto = str(texto).lower()
    texto = re.sub(r'@\w+', ' ', texto)
    texto = re.sub(r'http\S+|www\S+', ' ', texto)
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", texto)
    tokens = [re.sub(r"(\w){2,}", r"", tk) for tk in tokens]
    return " ".join(tokens)

## 4. Construccion del conjunto de entrenamiento y validacion

- Se limpia todo el dataset grande y luego se hace el split train/valid (0 y 4) con estratificacion.
- El dataset chico queda solo para validacion externa con 3 clases (0/2/4).
- Para recuperar la clase neutral en el set externo se usa una banda de confianza (0.4 - 0.6) sobre las probabilidades.


### Notas sobre la clase neutral (2)

Como el dataset grande no tiene clase 2, los modelos se entrenan en 0/4. En el set externo las probabilidades intermedias se mapearon a clase 2 para no forzar una decision positiva/negativa.


In [7]:
# Split sobre el dataset grande y guardo aparte el set externo chico
X_full_raw = big_df['text']
y_full = big_df['sentiment']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_full_raw, y_full, test_size=0.2, random_state=7, stratify=y_full
)

X_ext_raw = ext_df['text']
y_ext = ext_df['sentiment']

# limpieza aplicada por subconjunto (post-split)
X_train_clean = X_train_raw.apply(limpiar_tweet)
X_test_clean  = X_test_raw.apply(limpiar_tweet)
X_ext_clean   = X_ext_raw.apply(limpiar_tweet)

print("train:", X_train_clean.shape, "test:", X_test_clean.shape, "ext:", X_ext_clean.shape)


train: (1280000,) test: (320000,) ext: (498,)


## 5. Vectorización con TF-IDF

Para poder entrenar un modelo de clasificación, es necesario transformar el texto en vectores numéricos.

**TF-IDF**:

- TF (Term Frequency): frecuencia de aparición de un término en un documento.  
- IDF (Inverse Document Frequency): penaliza términos que aparecen en muchos documentos.  

De esta forma, cada tweet se representa como un vector de pesos TF-IDF.


### Vectorización corregida

In [ ]:
tfidf_vec = TfidfVectorizer(max_features=60000, ngram_range=(1,2))

X_train_tfidf = tfidf_vec.fit_transform(X_train_clean)
X_test_tfidf  = tfidf_vec.transform(X_test_clean)
X_ext_tfidf   = tfidf_vec.transform(X_ext_clean)

X_train_tfidf.shape, X_test_tfidf.shape, X_ext_tfidf.shape


## 6. Modelos supervisados para analisis de sentimiento

Se entrenan y evaluan dos modelos clasicos vistos en clase sobre el mismo conjunto de features (TF-IDF):

1. Logistic Regression (banda neutral para 3 clases en el set externo).
2. Multinomial Naive Bayes.

Se reportan metricas en la validacion interna (0/4) y en la externa (0/2/4).


### 6.1. Modelo 1 — Logistic Regression

In [ ]:
logreg = LogisticRegression(
    max_iter=300,
    multi_class='auto',
    solver='lbfgs',
    n_jobs=-1,
    class_weight='balanced'
)

logreg.fit(X_train_tfidf, y_train)

train_pred_lr = logreg.predict(X_train_tfidf)
test_pred_lr  = logreg.predict(X_test_tfidf)

print('Entrenamiento (0/4):')
print(classification_report(y_train, train_pred_lr, labels=[0,4]))
print('---')
print('Test interno (0/4):')
print(classification_report(y_test, test_pred_lr, labels=[0,4]))

low_thr, high_thr = 0.4, 0.6
proba_pos_lr = logreg.predict_proba(X_ext_tfidf)[:,1]
ext_pred_lr = np.where(proba_pos_lr > high_thr, 4,
                np.where(proba_pos_lr < low_thr, 0, 2))

print('Validacion externa (0/2/4):')
print(classification_report(y_ext, ext_pred_lr))

acc_lr_train = accuracy_score(y_train, train_pred_lr)
acc_lr_test  = accuracy_score(y_test, test_pred_lr)
acc_lr_ext   = accuracy_score(y_ext,   ext_pred_lr)


### 6.2. Modelo 2 — Multinomial Naive Bayes

In [ ]:
nb_clf = MultinomialNB()
nb_clf.fit(X_train_tfidf, y_train)

train_pred_nb = nb_clf.predict(X_train_tfidf)
test_pred_nb  = nb_clf.predict(X_test_tfidf)

print('Entrenamiento NB (0/4):')
print(classification_report(y_train, train_pred_nb, labels=[0,4]))
print('---')
print('Test interno NB (0/4):')
print(classification_report(y_test, test_pred_nb, labels=[0,4]))

proba_pos_nb = nb_clf.predict_proba(X_ext_tfidf)[:,1]
ext_pred_nb = np.where(proba_pos_nb > high_thr, 4,
               np.where(proba_pos_nb < low_thr, 0, 2))

print('Validacion externa NB (0/2/4):')
print(classification_report(y_ext, ext_pred_nb))

acc_nb_train = accuracy_score(y_train, train_pred_nb)
acc_nb_test  = accuracy_score(y_test, test_pred_nb)
acc_nb_ext   = accuracy_score(y_ext,   ext_pred_nb)


### 6.3. Comparacion resumida de accuracy entre modelos


In [ ]:
pd.DataFrame({
    'modelo': ['Logistic Regression', 'Multinomial NB'],
    'acc_train_0_4': [acc_lr_train, acc_nb_train],
    'acc_test_0_4': [acc_lr_test, acc_nb_test],
    'acc_ext_0_2_4': [acc_lr_ext, acc_nb_ext]
})


## 7. Matriz de confusión (modelo supervisado elegido)

Para profundizar en los errores, se muestra la matriz de confusión del modelo supervisado que se considere **más representativo** (por ejemplo, el de mayor accuracy entre Logistic Regression y Naive Bayes).


In [ ]:
labels_ext = [0,2,4]
cm = confusion_matrix(y_ext, ext_pred_lr, labels=labels_ext)

fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap='Oranges')
ax.set_xticks(range(len(labels_ext)))
ax.set_yticks(range(len(labels_ext)))
ax.set_xticklabels(labels_ext)
ax.set_yticklabels(labels_ext)
ax.set_xlabel('Predicho')
ax.set_ylabel('Real')
ax.set_title('Matriz de confusion - LogReg (externa)')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i,j], ha='center', va='center', color='black')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


In [ ]:
labels = [0,2,4]
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap='Blues')

ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)
ax.set_xlabel('Predicción')
ax.set_ylabel('Real')
ax.set_title('Matriz de confusión — Logistic Regression (3 clases)')

for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='black')

plt.tight_layout()
plt.show()

## 8. Comparacion con modelo pre-entrenado - TextBlob

La consigna sugiere comparar los modelos entrenados con al menos un modelo pre-entrenado.
Se usa TextBlob y se mapea la polaridad [-1,1] a las tres clases 0/2/4 con umbrales +/-0.1.


In [ ]:
def polarity_to_label(p):
    if p < -0.1:
        return 0
    elif p > 0.1:
        return 4
    else:
        return 2

textblob_scores = ext_df['text'].apply(lambda t: TextBlob(str(t)).sentiment.polarity)
textblob_pred = textblob_scores.apply(polarity_to_label)

print('Reporte de clasificacion - TextBlob (externa 0/2/4):')
print(classification_report(y_ext, textblob_pred))


### 8.1. Comparación resumida: modelos entrenados vs TextBlob

In [ ]:
acc_textblob = accuracy_score(y_ext, textblob_pred)

pd.DataFrame({
    'modelo': ['Logistic Regression', 'Multinomial NB', 'TextBlob (pre-entrenado)'],
    'acc_ext_0_2_4': [acc_lr_ext, acc_nb_ext, acc_textblob]
})


## 9. Metrica adicional: similitud de coseno entre tweets


In [ ]:
# Seleccionamos ejemplos de cada clase del set externo (etiquetado manual)
ej_neg = ext_df[ext_df['sentiment'] == 0].iloc[0]
ej_neu = ext_df[ext_df['sentiment'] == 2].iloc[0]
ej_pos = ext_df[ext_df['sentiment'] == 4].iloc[0]

print('Tweet NEGATIVO:', ej_neg['text'], '')
print('Tweet NEUTRAL:', ej_neu['text'], '')
print('Tweet POSITIVO:', ej_pos['text'], '')

# Vectorizamos los tres ejemplos ya limpiados
examples_clean = [
    limpiar_tweet(ej_neg['text']),
    limpiar_tweet(ej_neu['text']),
    limpiar_tweet(ej_pos['text'])
]
X_examples = tfidf_vec.transform(examples_clean)

sim_matrix = cosine_similarity(X_examples)
labels_sim = ['negativo', 'neutral', 'positivo']
sim_df = pd.DataFrame(sim_matrix, index=labels_sim, columns=labels_sim)
print('Matriz de similitud de coseno entre ejemplos:')
display(sim_df)

fig, ax = plt.subplots(figsize=(4,3))
im = ax.imshow(sim_matrix, cmap='Purples')
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(labels_sim)
ax.set_yticklabels(labels_sim)
ax.set_title('Similitud de coseno entre tweets de ejemplo')

for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha='center', va='center', color='black')

plt.tight_layout()
plt.show()


## 10. Conclusiones

1) Limpieza aplicada despues del split para evitar sesgos: dataset grande -> train/test; dataset chico solo para validacion externa.
2) TF-IDF (1-2 ngramas) + Logistic Regression con banda neutral rindio mejor que Naive Bayes en el set externo.
3) TextBlob se mantiene como baseline preentrenado y queda por debajo de los modelos entrenados.
